# EFS-04 resource addendum

This additive notebook preserves the original notebook and adds source-backed validation tables for quantitative momentum II.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['efs04_source_zip_selection.csv','efs04_strategy_inventory.csv','efs04_timestamp_controls.csv','efs04_vix_es_hedge_reference.csv','efs04_momentum_crash_controls.csv','efs04_autocorrelation_iid_demo.csv','efs04_calendar_levered_etf_reference.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Source inventory

The copied source packet includes the oil, VIX/ES, Treasury-note momentum, VIX cash-index, and raw MATLAB data artifacts used by the lecture family.

In [ ]:
source = data['efs04_source_zip_selection.csv']
print(source[['copied_as','role','bytes']].to_string(index=False))
assert source['copied_as'].str.contains('VX_ES_rollreturn_source.py').any()
assert source['copied_as'].str.contains('VIX_source.csv').any()
assert source['copied_as'].str.contains('TU_mom_source.py').any()

## 2. Strategy inventory and data hygiene

The lecture strategies are not just chart patterns. Each one has a data, hedge, funding, or regime control.

In [ ]:
strategies = data['efs04_strategy_inventory.csv']
print(strategies[['strategy','signal','lecture_control']].to_string(index=False))
assert strategies['strategy'].str.contains('VIX futures roll').any()
assert strategies['strategy'].str.contains('Leveraged ETF').any()

## 3. Timestamp and hedge-selection corrections

Back-adjustment fixes futures roll gaps, but not cross-market timestamp mismatch. USO and XLE also do different jobs in the crude trade.

In [ ]:
ts = data['efs04_timestamp_controls.csv']
print(ts.to_string(index=False))
assert ts['correct_version'].str.contains('2:30pm').any()
assert ts['correct_version'].str.contains('does not fix timestamp').any()

## 4. VIX/ES implementation reference

The VIX index is a reference, VX is the traded future, and ES is the same-side hedge because VIX and ES are anti-correlated.

In [ ]:
vix = data['efs04_vix_es_hedge_reference.csv']
print(vix[['component','source_rule','caveat']].to_string(index=False))
assert vix['source_rule'].str.contains('0.315377').any()
assert vix['source_rule'].str.contains('same-side').any()

## 5. Autocorrelation test discipline

The overlapping sample can report a p-value that looks far more confident than the IID-corrected sample.

In [ ]:
auto = data['efs04_autocorrelation_iid_demo.csv']
print(auto[['sample_method','sample_count','correlation','p_value_approx','iid_valid']].to_string(index=False))
assert auto.loc[0, 'sample_count'] > auto.loc[1, 'sample_count']
assert auto.loc[1, 'iid_valid'] == 'yes'

## 6. Momentum-crash, calendar-spread, and leveraged-ETF controls

These tables separate the tradable idea from the implementation trap.

In [ ]:
crash = data['efs04_momentum_crash_controls.csv']
cal = data['efs04_calendar_levered_etf_reference.csv']
print(crash[['issue','fix']].to_string(index=False))
print(cal[['topic','trading_implication']].to_string(index=False))
assert crash['fix'].str.contains('mean reversion', case=False).any()
assert cal['lecture_rule'].str.contains('linear curve', case=False).any()
assert cal['topic'].str.contains('Leveraged ETF drag').any()